In [1]:
import shutil
import os

In [2]:
from zipfile import ZipFile
from pathlib import Path
import glob
import json


In [3]:
results_path = "../data/results_test/*.json"
merged_path = "../data/merge_test/*.json"
results = glob.glob(results_path)
results = [Path(p) for p in results]
merged = glob.glob(merged_path)
merged = [Path(p) for p in merged]
onto_path = Path("../data/results_onto/Graphwise_GutBrainIE_2026")

In [4]:
results, merged

([PosixPath('../data/results_test/eval_naive-entities.json'),
  PosixPath('../data/results_test/eval_hermes_neefinetuned-rag-lora-entities-naive-beam-none-hermes-3-2-3B-entities.json'),
  PosixPath('../data/results_test/eval_hermes-rag-beam-none-hermes-3-2-3B-base.json'),
  PosixPath('../data/results_test/eval_hermes-rag-beam-none-hermes-3-1-8B-base.json'),
  PosixPath('../data/results_test/eval_hermes-lora-relations-naive-beam-none-hermes-3-1-8B-relations.json'),
  PosixPath('../data/results_test/eval_hermes_neefinetuned-lora-entities-naive-beam-none-hermes-3-2-3B-entities.json'),
  PosixPath('../data/results_test/eval_hermes_neefinetuned-rag-lora-relations-naive-beam-none-hermes-3-2-3B-relations.json'),
  PosixPath('../data/results_test/eval_hermes_neefinetuned-rag-lora-entities-beam-none-hermes-3-1-8B-entities.json'),
  PosixPath('../data/results_test/eval_hermes_neefinetuned-rag-beam-none-hermes-3-2-3B-base.json'),
  PosixPath('../data/results_test/eval_hermes_neefinetuned-rag-beam

In [5]:
removed_ids= [
    "eval_hermes_neefinetuned-rag-lora-relations-beam-none-",
    "eval_hermes_neefinetuned-lora-relations-beam-none-",
    "eval_hermes_neefinetuned-rag-lora-entities-beam-none-",
    "eval_hermes_neefinetuned-lora-entities-beam-none-",
    "eval_hermes-lora-entities-beam-none-",
]
for run_id in removed_ids:
    to_remove_results = [p for p in results if run_id in p.name]
    to_remove_merged = [p for p in merged if run_id in p.name]
    for p in to_remove_results:
        print(f"Removing {p} from results")
    for p in to_remove_merged:
        print(f"Removing {p} from merged")
    results = [p for p in results if run_id not in p.name]
    merged = [p for p in merged if run_id not in p.name]

Removing ../data/results_test/eval_hermes_neefinetuned-rag-lora-relations-beam-none-hermes-3-2-3B-relations.json from results
Removing ../data/results_test/eval_hermes_neefinetuned-rag-lora-relations-beam-none-hermes-3-1-8B-relations.json from results
Removing ../data/results_test/eval_hermes_neefinetuned-lora-relations-beam-none-hermes-3-1-8B-relations.json from results
Removing ../data/results_test/eval_hermes_neefinetuned-lora-relations-beam-none-hermes-3-2-3B-relations.json from results
Removing ../data/results_test/eval_hermes_neefinetuned-rag-lora-entities-beam-none-hermes-3-1-8B-entities.json from results
Removing ../data/results_test/eval_hermes_neefinetuned-rag-lora-entities-beam-none-hermes-3-2-3B-entities.json from results
Removing ../data/results_test/eval_hermes_neefinetuned-lora-entities-beam-none-hermes-3-1-8B-entities.json from results
Removing ../data/results_test/eval_hermes_neefinetuned-lora-entities-beam-none-hermes-3-2-3B-entities.json from results
Removing ../data

In [6]:
task_ids = {
    "T611": "entities",
    "T612": "entities",
    "T621": "mention_level_relations",
    "T622": "concept_level_relations",
}
allowed_keys = {
    "T611": ["start_idx", "end_idx", "location", "text_span", "label"],
    "T612": ["start_idx", "end_idx", "location", "text_span", "label", "uri"],
    "T621": [
        "subject_text_span",
        "subject_label",
        "predicate",
        "object_text_span",
        "object_label",
    ],
    "T622": ["subject_label", "subject_uri", "predicate", "object_label", "object_uri"],
}
run_ids = [str(p.name).split(".")[0] for p in results]
system_id = "CHASTE"
team_id = "ToGS"
run_ids

['eval_naive-entities',
 'eval_hermes_neefinetuned-rag-lora-entities-naive-beam-none-hermes-3-2-3B-entities',
 'eval_hermes-rag-beam-none-hermes-3-2-3B-base',
 'eval_hermes-rag-beam-none-hermes-3-1-8B-base',
 'eval_hermes-lora-relations-naive-beam-none-hermes-3-1-8B-relations',
 'eval_hermes_neefinetuned-lora-entities-naive-beam-none-hermes-3-2-3B-entities',
 'eval_hermes_neefinetuned-rag-lora-relations-naive-beam-none-hermes-3-2-3B-relations',
 'eval_hermes_neefinetuned-rag-beam-none-hermes-3-2-3B-base',
 'eval_hermes_neefinetuned-rag-beam-none-hermes-3-1-8B-base',
 'eval_hermes-rag-lora-relations-beam-none-hermes-3-1-8B-relations',
 'eval_hermes-rag-lora-entities-beam-none-hermes-3-2-3B-entities',
 'eval_hermes-rag-lora-entities-naive-beam-none-hermes-3-2-3B-entities',
 'eval_hermes-rag-lora-relations-naive-beam-none-hermes-3-1-8B-relations',
 'eval_hermes-lora-relations-beam-none-hermes-3-1-8B-relations',
 'eval_hermes-rag-naive-beam-none-hermes-3-2-3B-base',
 'eval_hermes-rag-naive

In [7]:
onto_path

PosixPath('../data/results_onto/Graphwise_GutBrainIE_2026')

In [8]:
list(onto_path.glob("*"))

[PosixPath('../data/results_onto/Graphwise_GutBrainIE_2026/Graphwise_T621_BQWEN_qwen354b'),
 PosixPath('../data/results_onto/Graphwise_GutBrainIE_2026/Graphwise_T611_13_NER-EXP-8-SUB-1'),
 PosixPath('../data/results_onto/Graphwise_GutBrainIE_2026/Graphwise_T612_1_BaseFreqGaze'),
 PosixPath('../data/results_onto/Graphwise_GutBrainIE_2026/Graphwise_T612_5_22-ENS-EXP-3-SUB-1-EL-freq-gaze'),
 PosixPath('../data/results_onto/Graphwise_GutBrainIE_2026/.DS_Store'),
 PosixPath('../data/results_onto/Graphwise_GutBrainIE_2026/Graphwise_T611_22_ENS-EXP-3-SUB-1'),
 PosixPath('../data/results_onto/Graphwise_GutBrainIE_2026/Graphwise_T611_18_GLINER-EXP-2-SUB-1'),
 PosixPath('../data/results_onto/Graphwise_GutBrainIE_2026/Graphwise_T622_BGPT5520G_gpt-5.5_20_shot_gold'),
 PosixPath('../data/results_onto/Graphwise_GutBrainIE_2026/Graphwise_T622_BQWEN_qwen354b'),
 PosixPath('../data/results_onto/Graphwise_GutBrainIE_2026/Graphwise_T612_2_24-ENS-EXP-4-SUB-1-EL-freq-gaze'),
 PosixPath('../data/results_ont

In [9]:

merge_run_ids = [p.name[:-len(".json")] for p in merged]
merge_run_ids

['eval_hermes_neefinetuned-lora-entities-naive-beam-none-hermes-3-2-3B-entities_Graphwise_T622_BGPT5520G_gpt-5.5_20_shot_gold_intersection',
 'eval_hermes_neefinetuned-lora-entities-naive-beam-none-hermes-3-2-3B-entities_Graphwise_T621_BGPT5520G_gpt-5.5_20_shot_gold_union',
 'eval_hermes-lora-relations-beam-none-hermes-3-2-3B-relations_Graphwise_T611_24_ENS-EXP-4-SUB-1_union',
 'eval_hermes-lora-relations-beam-none-hermes-3-2-3B-relations_Graphwise_T621_BGPT5520G_gpt-5.5_20_shot_gold_intersection',
 'eval_naive-entities_Graphwise_T612_2_24-ENS-EXP-4-SUB-1-EL-freq-gaze_union',
 'eval_naive-entities_Graphwise_T621_BGPT5520G_gpt-5.5_20_shot_gold_union',
 'eval_naive-entities_Graphwise_T611_24_ENS-EXP-4-SUB-1_union',
 'eval_naive-entities_Graphwise_T621_BGPT5520G_gpt-5.5_20_shot_gold_intersection',
 'eval_hermes-lora-relations-beam-none-hermes-3-2-3B-relations_Graphwise_T612_3_13-NER-EXP-8-SUB-1-EL-freq-gaze_union',
 'eval_hermes_neefinetuned-lora-entities-naive-beam-none-hermes-3-2-3B-ent

In [10]:
import chevron
import json

In [11]:
staging_dir = Path("./staging")
import re
import shutil

shutil.rmtree(staging_dir, ignore_errors=True)
zip_folders: list[Path] = []
staging_dir.mkdir(exist_ok=True)


def stage_run(task_id: str, run_id: str, result_path: Path):
    team_id_used = team_id
    print(f"Processing {task_id} - {run_id}... ({result_path})")

    run_id_simples = (
        run_id.replace("-", "")
        .replace("eval_", "")
        .replace("relation", "")
        .replace("entity", "")
    )
    run_id_simples = re.sub(r"_|\\.", "", run_id_simples)
    system_id_used = system_id
    desc_data = ""
    additional_desc = ""
    with open("./description.md", "r") as f:
        desc_data = f.read(-1)
    flags = []
    if "rag" in run_id:
        flags.append("RAG")
    if "reorder" in run_id:
        flags.append("Reordered")
    if "neefinetune" in run_id:
        flags.append("Finetuned Named Entity Extraction")
    if "lora" in run_id:
        flags.append("Finetuned using LoRA")
    if "hermes-3-2-3B" in run_id:
        flags.append("Base Model: Hermes-3-2-3B")
    if "hermes-3-1-8B" in run_id:
        flags.append("Base Model: Hermes-3-1-8B")
    if "eval_naive-filtered" in run_id:
        flags.append("Run with naive approach (filtered)")
    if "beam-none" in run_id:
        flags.append("Run without beam search")
    if "beam-end" in run_id:
        flags.append("Run with beam search up to end token")
    if "beam-shallow" in run_id:
        flags.append("Run with shallow beam search for each token")
    elif "eval_naive" in run_id:
        flags.append("Run with naive approach")
        system_id_used = "Naive"
    if "Graphwise" in run_id:
        if "intersection" in run_id:
            flags.append(
                "Merged with Graphwise results (intersection), their run descrription is appended"
            )
        elif "union" in run_id:
            flags.append(
                "Merged with Graphwise results (union), their run descrription is appended"
            )
        system_id_used = system_id_used + "Gw"
        team_id_used = "TUGW"
        # load metada from Graphwise run
        gw = "Graphwise"
        onto_run_id = re.search(rf"({gw}.*)_(intersection|union)$", run_id).group(1)
        # run_id_simples = re.sub(r"_Graphwise.*$", "Gw", run_id_simples)
        graphwise_desc_path = onto_path / onto_run_id / f"{onto_run_id}.meta"
        with open(graphwise_desc_path, "r") as f:
            additional_desc = (
                f"\n\n### Additional description from {gw} run:\n\n" + f.read(-1)
            )

    rendered_desc = chevron.render(
        desc_data,
        {
            "task_id": task_id,
            "run_id": run_id,
            "system_id": system_id_used,
            "team_id": team_id_used,
            "flags": flags,
        },
    )
    rendered_desc += additional_desc
    run_data: dict[str, dict[str, any]] = None
    with open(result_path, "r") as rf:
        run_data = json.load(rf)
    stratified_res = {}
    all_empty = True
    for k, res in run_data.items():
        if task_ids[task_id] not in res:
            continue
        predictions: list[dict[str, str | int]] = res[task_key]
        predictions = [
            {k: v for k, v in p.items() if k in allowed_keys[task_id]}
            for p in predictions
        ]
        if len(predictions) > 0:
            all_empty = False
        stratified_res[k] = {task_key: predictions}

    identifier = f"{team_id_used}_{task_id}_{run_id_simples}_{system_id_used}"
    identifier_dir = staging_dir / identifier
    if all_empty:
        print(
            f"Warning: All predictions are empty for {identifier} on {run_id}- skipping submission generation."
        )
        return
    identifier_dir.mkdir(exist_ok=True)
    desc_file = identifier_dir / f"{identifier}.meta"
    out_file = identifier_dir / f"{identifier}.json"
    with open(desc_file, "w") as f:
        f.write(rendered_desc)
    with open(out_file, "w") as rf:
        json.dump(stratified_res, rf, indent=2)
    zip_folders.append(identifier_dir)


for task_id, task_key in task_ids.items():
    for run_id, result_path in zip(merge_run_ids, merged):
        stage_run(task_id, run_id, result_path)
    for run_id, result_path in zip(run_ids, results):
        stage_run(task_id, run_id, result_path)
        # zip_path = staging_dir / f"{identifier}.zip"
        # zf = ZipFile(zip_path, "w")
        # zf.write(desc_file, f"{identifier}.md")
        # zf.write(out_file, f"{identifier}.json")

Processing T611 - eval_hermes_neefinetuned-lora-entities-naive-beam-none-hermes-3-2-3B-entities_Graphwise_T622_BGPT5520G_gpt-5.5_20_shot_gold_intersection... (../data/merge_test/eval_hermes_neefinetuned-lora-entities-naive-beam-none-hermes-3-2-3B-entities_Graphwise_T622_BGPT5520G_gpt-5.5_20_shot_gold_intersection.json)
Processing T611 - eval_hermes_neefinetuned-lora-entities-naive-beam-none-hermes-3-2-3B-entities_Graphwise_T621_BGPT5520G_gpt-5.5_20_shot_gold_union... (../data/merge_test/eval_hermes_neefinetuned-lora-entities-naive-beam-none-hermes-3-2-3B-entities_Graphwise_T621_BGPT5520G_gpt-5.5_20_shot_gold_union.json)


Processing T611 - eval_hermes-lora-relations-beam-none-hermes-3-2-3B-relations_Graphwise_T611_24_ENS-EXP-4-SUB-1_union... (../data/merge_test/eval_hermes-lora-relations-beam-none-hermes-3-2-3B-relations_Graphwise_T611_24_ENS-EXP-4-SUB-1_union.json)
Processing T611 - eval_hermes-lora-relations-beam-none-hermes-3-2-3B-relations_Graphwise_T621_BGPT5520G_gpt-5.5_20_shot_gold_intersection... (../data/merge_test/eval_hermes-lora-relations-beam-none-hermes-3-2-3B-relations_Graphwise_T621_BGPT5520G_gpt-5.5_20_shot_gold_intersection.json)
Processing T611 - eval_naive-entities_Graphwise_T612_2_24-ENS-EXP-4-SUB-1-EL-freq-gaze_union... (../data/merge_test/eval_naive-entities_Graphwise_T612_2_24-ENS-EXP-4-SUB-1-EL-freq-gaze_union.json)
Processing T611 - eval_naive-entities_Graphwise_T621_BGPT5520G_gpt-5.5_20_shot_gold_union... (../data/merge_test/eval_naive-entities_Graphwise_T621_BGPT5520G_gpt-5.5_20_shot_gold_union.json)
Processing T611 - eval_naive-entities_Graphwise_T611_24_ENS-EXP-4-SUB-1_unio

In [12]:

zip_path = staging_dir.parent / f"{team_id}_GutBrainIE_2026.zip"
with ZipFile(zip_path, "w") as zf:
    for identifier_dir in zip_folders:
        for file in identifier_dir.iterdir():
            zf.write(file, f"{identifier_dir.name}/{file.name}")

In [13]:
for id in task_ids.keys():
    task_ids_subms = list(staging_dir.glob(f"*_{id}_*"))
    team_ids = set([p.name.split("_")[0] for p in task_ids_subms])
    for team_id in team_ids:
        team_subms = list(staging_dir.glob(f"{team_id}_{id}_*"))
        print(f"Team {team_id} has {len(list(team_subms))} submissions for task {id}.")
    print(f"Task {id} has {len(list(task_ids_subms))} submissions.")

Team ToGS has 24 submissions for task T611.
Team TUGW has 16 submissions for task T611.
Task T611 has 40 submissions.
Team ToGS has 24 submissions for task T612.
Team TUGW has 16 submissions for task T612.
Task T612 has 40 submissions.
Team ToGS has 25 submissions for task T621.
Task T621 has 25 submissions.
Team ToGS has 25 submissions for task T622.
Team TUGW has 7 submissions for task T622.
Task T622 has 32 submissions.
